# 避難所候補算出支援ツール（sheltermatch）

要支援者一覧と避難所一覧の座標から、要支援者ごとに **距離が近い避難所候補（既定で上位3件）** を算出し、
必要であればハザード区域との位置関係も確認して、職員の最終判断のための資料（CSV）を作成するNotebookです。

**このツールが行うこと**
- 直線距離（`geopy.distance.geodesic`）による避難所候補の算出（候補提示。自動割当ではありません）
- 任意機能として、要支援者・避難所・両者を結ぶ直線とハザード区域（GeoJSON）との位置関係の確認

**このツールが行わないこと（重要）**
- 避難先・避難経路の自動決定
- 道路経路・通行可能性・坂道・高低差・災害時の道路状況の計算（ダイクストラ法や道路ネットワーク、Directions API等は使用しません）
- ハザード判定結果による候補避難所の自動除外・自動順位変更
- 独自の危険度スコアリング

距離順位とハザード判定は別々の情報として出力します。どの避難所を選ぶかは、出力結果を確認した **職員が判断** してください。

**個人情報の取り扱い**
- 実際の要支援者データ・ハザードデータはこのリポジトリにコミットしないでください（サンプルを追加する場合は完全な架空データを使用してください）。
- 出力CSVには住所・座標等の個人情報が含まれ得ます。取り扱いに注意してください。
- 住所→座標変換にはローカル辞書を使う **Jageocoder** のみを使用し、Google Maps API・OpenStreetMap Nominatim等の外部公開ジオコーディングサービスへ実住所を送信しません。

上から順にセルを実行すれば処理が完了します。コードを読み込まなくても、各セルの説明とprint出力で操作できます。

## 1. 必要ライブラリの準備

Google Colabに標準で入っていないライブラリをインストールします。初回のみ数十秒かかることがあります。

In [ ]:
%pip install -q geopy jageocoder geopandas shapely

In [ ]:
import io

import numpy as np
import pandas as pd
from geopy.distance import geodesic

import geopandas as gpd
from shapely.geometry import Point, LineString

from google.colab import files

print("ライブラリの読み込みが完了しました。")

## 2. 設定

このNotebookで編集が必要な設定はこのセルだけです。値を変更してから実行してください。

In [ ]:
# ===== 設定（このセルの値を必要に応じて編集してください） =====

# 住所→座標変換（Jageocoderのローカル辞書によるジオコーディング）を使うかどうか
# True にする場合は、あらかじめ用意したJageocoderのローカル辞書が必要です。
ENABLE_GEOCODING = False

# Jageocoderのローカル辞書ディレクトリのパス（ENABLE_GEOCODING=True の場合のみ使用）
# 例: Google Driveをマウントした場合 "/content/drive/MyDrive/jageocoder_db"
JAGEOCODER_DB_DIR = ""

# Jageocoderの一致レベルの下限（この値未満は「粗い一致」とみなし、座標を自動採用せず確認対象とする）
# level はJageocoderの定義で 7=街区・地番(Block), 8=建物(Building) 等を示す値です
# （このNotebookの一次候補算出には、街区・地番レベル以上の座標で十分と考え既定値7としています。
#  丁目程度までしか一致しなかった粗い結果を自動採用しないための下限値であり、
#  住所文字列全体が完全一致したことまでは保証しません）。運用に応じて調整してください。
JAGEOCODER_MIN_LEVEL = 7

# ハザード区域（GeoJSON）との位置関係を確認するかどうか
ENABLE_HAZARD_CHECK = False

# 要支援者ごとに算出する避難所候補の件数（既定3件。避難所がこの件数未満の場合は存在する件数まで出力）
TOP_N = 3

# 出力するCSVファイル名
OUTPUT_FILENAME = "assigned_shelters.csv"

if not isinstance(TOP_N, int) or TOP_N < 1:
    raise ValueError(f"TOP_N は1以上の整数を指定してください。現在の値: {TOP_N!r}")

print("設定を読み込みました。")
print(f"  ENABLE_GEOCODING     = {ENABLE_GEOCODING}")
print(f"  JAGEOCODER_DB_DIR    = '{JAGEOCODER_DB_DIR}'")
print(f"  JAGEOCODER_MIN_LEVEL = {JAGEOCODER_MIN_LEVEL}")
print(f"  ENABLE_HAZARD_CHECK  = {ENABLE_HAZARD_CHECK}")
print(f"  TOP_N                = {TOP_N}")
print(f"  OUTPUT_FILENAME      = '{OUTPUT_FILENAME}'")

## 3. Google Driveのマウント（任意）

Jageocoderの辞書をGoogle Drive上に置いている場合のみ実行してください。
辞書を使わない場合や、既にColab環境上に辞書がある場合はこのセルは不要です（実行しなくても後続処理に影響しません）。

In [ ]:
# 必要な場合のみ True に変更してこのセルを実行してください。
MOUNT_GOOGLE_DRIVE = False

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    print("Google Driveをマウントしました。")
else:
    print("Google Driveのマウントをスキップしました。")

## 4. CSVファイルのアップロード

「要支援者一覧CSV」と「避難所一覧CSV」を、それぞれのセルの指示に従って選択してください。
ファイル名は自由です（Notebook内でファイル名を固定していません）。

In [ ]:
print("【要支援者一覧CSV】を選択してください。")
uploaded_residents = files.upload()

if len(uploaded_residents) != 1:
    raise RuntimeError("要支援者一覧CSVは1つだけ選択してください。")

residents_filename = list(uploaded_residents.keys())[0]
residents_bytes = uploaded_residents[residents_filename]
print(f"'{residents_filename}' を要支援者一覧として受け取りました。")

In [ ]:
print("【避難所一覧CSV】を選択してください。")
uploaded_shelters = files.upload()

if len(uploaded_shelters) != 1:
    raise RuntimeError("避難所一覧CSVは1つだけ選択してください。")

shelters_filename = list(uploaded_shelters.keys())[0]
shelters_bytes = uploaded_shelters[shelters_filename]
print(f"'{shelters_filename}' を避難所一覧として受け取りました。")

## 5. CSV読込・入力チェック

文字コードを自動判定して読み込みます（UTF-8 BOM付き → CP932 → UTF-8 の順で試行）。
続いて座標列（`latitude` / `longitude`）が数値として扱えるか、緯度が-90〜90、経度が-180〜180の範囲かを確認します。
座標が欠損・不正な要支援者の行があっても削除せず、後続の距離計算のみ対象外とします。
`ENABLE_GEOCODING = True` の場合は、`latitude` / `longitude` 列が無く `address` 列のみのCSVも読み込めます
（次のセクションでジオコーディングして座標を補完します）。

In [ ]:
def read_csv_auto(file_bytes, label):
    """UTF-8(BOM付き) → CP932 → UTF-8 の順で読み込みを試み、成功したDataFrameを返す。"""
    encodings = [
        ("utf-8-sig", "UTF-8 (BOM付き)"),
        ("cp932", "CP932 (Shift-JIS系)"),
        ("utf-8", "UTF-8"),
    ]
    last_error = None
    for encoding, encoding_label in encodings:
        try:
            df = pd.read_csv(io.BytesIO(file_bytes), encoding=encoding)
            print(f"[{label}] {encoding_label} として読み込みました。（{len(df)}行）")
            return df
        except (UnicodeDecodeError, UnicodeError) as error:
            last_error = error
            continue
    raise ValueError(
        f"[{label}] 文字コードを判定できませんでした。"
        "UTF-8(BOM付き)・CP932・UTF-8のいずれでも読み込めません。"
        "Excel等での保存時の文字コードを確認してください。"
        f" 詳細: {last_error}"
    )


def require_columns(df, required_columns, label):
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"[{label}] 必須列が見つかりません: {', '.join(missing)}")


def ensure_coordinate_columns(df, label):
    """latitude/longitude列が無い場合、ENABLE_GEOCODING=True かつ address列があれば
    ジオコーディング対象として空の座標列を追加する。それ以外は列不足として停止する。"""
    df = df.copy()
    if "latitude" in df.columns and "longitude" in df.columns:
        return df

    if ENABLE_GEOCODING and "address" in df.columns:
        df["latitude"] = np.nan
        df["longitude"] = np.nan
        print(f"[{label}] latitude/longitude列が無いため、address列からのジオコーディング対象として空の座標列を追加しました。")
        return df

    missing = [c for c in ("latitude", "longitude") if c not in df.columns]
    hint = "" if ENABLE_GEOCODING else "（ENABLE_GEOCODING=Trueにするとaddress列のみでも処理できます）"
    raise ValueError(f"[{label}] 必須列が見つかりません: {', '.join(missing)}{hint}")


residents_raw = read_csv_auto(residents_bytes, "要支援者一覧")
shelters_raw = read_csv_auto(shelters_bytes, "避難所一覧")

residents_raw = ensure_coordinate_columns(residents_raw, "要支援者一覧")
shelters_raw = ensure_coordinate_columns(shelters_raw, "避難所一覧")

require_columns(shelters_raw, ["name"], "避難所一覧")

display(residents_raw.head())
display(shelters_raw.head())

In [ ]:
def parse_and_validate_coordinates(df, label):
    """latitude/longitude列を数値化し、有効な座標かどうかの真偽値Seriesを返す。"""
    lat = pd.to_numeric(df["latitude"], errors="coerce")
    lon = pd.to_numeric(df["longitude"], errors="coerce")
    valid = lat.notna() & lon.notna() & lat.between(-90, 90) & lon.between(-180, 180)
    invalid_count = int((~valid).sum())
    if invalid_count:
        print(f"[{label}] 座標が欠損・不正な行が {invalid_count}件あります（全{len(df)}行中）。")
    return lat, lon, valid


residents = residents_raw.copy()
residents["latitude"], residents["longitude"], residents_coord_valid = parse_and_validate_coordinates(
    residents, "要支援者一覧"
)

shelters = shelters_raw.copy()
shelters["latitude"], shelters["longitude"], shelters_coord_valid = parse_and_validate_coordinates(
    shelters, "避難所一覧"
)

## 6. 必要に応じた住所→座標変換（Jageocoder）

`ENABLE_GEOCODING = True` の場合のみ、座標が空欄で `address` 列がある行について、
Jageocoderのローカル辞書で住所検索を行い座標を補完します。既に座標がある行はそのまま使用します。
外部の公開ジオコーディングサービスへは実住所を送信しません。
`ENABLE_GEOCODING = False`（既定）の場合はこのセルは何もせずスキップします。

Jageocoderは住所の途中まで（町字・丁目程度）しか一致しない場合でも候補を返すことがあります。
街区・地番レベル以上（`level >= JAGEOCODER_MIN_LEVEL`、既定7）まで一致しなかった行は **座標を自動採用せず**、
`geocode_status = "coarse_match"` として座標を空欄のまま残し、職員による住所確認の対象とします。
なお `level` は「どこまで一致したか」の目安であり、街区・地番レベルで一致した場合でも、
入力した住所文字列全体が完全に一致したことまでは保証しません（詳細は設定セルのコメントを参照）。
辞書検索中にエラーが発生した場合も `"not_found"` とは区別し、`"error"` として記録します。

In [ ]:
def init_jageocoder():
    import jageocoder

    if not JAGEOCODER_DB_DIR:
        raise RuntimeError(
            "ENABLE_GEOCODING=True ですが JAGEOCODER_DB_DIR が設定されていません。"
            "設定セルでJageocoderのローカル辞書ディレクトリのパスを指定してください。"
        )
    try:
        jageocoder.init(db_dir=JAGEOCODER_DB_DIR)
    except Exception as error:
        raise RuntimeError(
            "Jageocoderのローカル辞書の初期化に失敗しました。"
            f"JAGEOCODER_DB_DIR='{JAGEOCODER_DB_DIR}' が正しい辞書ディレクトリを指しているか確認してください。"
            f" 詳細: {error}"
        )
    return jageocoder


def geocode_address(jageocoder_module, address):
    """Jageocoderのローカル辞書で住所を検索し、結果を状態付きの辞書で返す。

    status: "matched"(詳細住所まで一致・座標採用) / "coarse_match"(粗い一致・座標は未採用)
            / "not_found"(該当なし) / "error"(検索処理中の例外)
    座標を推測することはせず、一致しない・粗い一致の場合は座標を返さない。
    """
    if not isinstance(address, str) or not address.strip():
        return {"status": "not_found"}

    try:
        results = jageocoder_module.searchNode(address.strip())
    except Exception as error:
        return {"status": "error", "error": str(error)}

    if not results:
        return {"status": "not_found"}

    # jageocoderの戻り値の形状はバージョンにより差異があるため、取得できる範囲のみ利用する
    node = results[0].node
    try:
        matched_name = node.get_fullname()
        if isinstance(matched_name, (list, tuple)):
            matched_name = "".join(matched_name)
    except Exception:
        matched_name = str(node)
    level = getattr(node, "level", None)

    if level is not None and level < JAGEOCODER_MIN_LEVEL:
        return {"status": "coarse_match", "matched": matched_name, "level": level}

    return {
        "status": "matched",
        "lat": node.y,
        "lon": node.x,
        "matched": matched_name,
        "level": level,
    }


def fill_missing_coordinates(df, coord_valid, label):
    """座標が空欄の行についてのみ、address列からジオコーディングして座標を補完する。
    粗い一致・該当なし・エラーの行は座標を採用せず、geocode_statusに状態を残す。"""
    if not ENABLE_GEOCODING:
        return df, coord_valid

    df = df.copy()
    if "address" not in df.columns:
        print(f"[{label}] address列が無いため、ジオコーディングを行いません。")
        return df, coord_valid

    jageocoder_module = init_jageocoder()

    # 文字列(status/matched)を後から代入するため、float64ではなくobject dtypeで列を用意する
    df["geocode_status"] = pd.Series(np.nan, index=df.index, dtype="object")
    df["geocode_matched"] = pd.Series(np.nan, index=df.index, dtype="object")
    df["geocode_level"] = np.nan

    missing_mask = df["latitude"].isna() | df["longitude"].isna()
    target_count = int(missing_mask.sum())
    status_counts = {"matched": 0, "coarse_match": 0, "not_found": 0, "error": 0}

    for idx in df.index[missing_mask]:
        result = geocode_address(jageocoder_module, df.at[idx, "address"])
        status = result["status"]
        status_counts[status] = status_counts.get(status, 0) + 1
        df.at[idx, "geocode_status"] = status

        if status == "matched":
            df.at[idx, "latitude"] = result["lat"]
            df.at[idx, "longitude"] = result["lon"]
            df.at[idx, "geocode_matched"] = result["matched"]
            df.at[idx, "geocode_level"] = result["level"]
        elif status == "coarse_match":
            df.at[idx, "geocode_matched"] = result["matched"]
            df.at[idx, "geocode_level"] = result["level"]

    print(
        f"[{label}] ジオコーディング対象 {target_count}件: "
        f"詳細一致(座標採用) {status_counts['matched']}件 / "
        f"粗い一致(要確認) {status_counts['coarse_match']}件 / "
        f"該当なし {status_counts['not_found']}件 / "
        f"エラー {status_counts['error']}件"
    )

    lat = pd.to_numeric(df["latitude"], errors="coerce")
    lon = pd.to_numeric(df["longitude"], errors="coerce")
    valid = lat.notna() & lon.notna() & lat.between(-90, 90) & lon.between(-180, 180)
    df["latitude"], df["longitude"] = lat, lon
    return df, valid


residents, residents_coord_valid = fill_missing_coordinates(residents, residents_coord_valid, "要支援者一覧")
shelters, shelters_coord_valid = fill_missing_coordinates(shelters, shelters_coord_valid, "避難所一覧")

if not ENABLE_GEOCODING:
    print("ENABLE_GEOCODING=False のため、住所→座標変換をスキップしました。")

## 7. ハザードデータの読込（任意）

`ENABLE_HAZARD_CHECK = True` の場合のみ、ハザード区域のGeoJSONファイルをアップロードします。
洪水・土砂災害・津波・高潮等、複数種類のGeoJSONをまとめて選択できます。ファイルごとにハザード種別名を入力してください。
座標系はWGS84（EPSG:4326）に統一され、Polygon/MultiPolygon以外の空・不正なジオメトリは除外されます。
`ENABLE_HAZARD_CHECK = False`（既定）の場合はこのセルはスキップされ、ハザードデータなしで距離候補算出のみが実行されます。

**注意**: `ENABLE_HAZARD_CHECK = True` にした場合、有効なハザード区域ポリゴンが1件も読み込めなかったときは
「ハザードなし」とみなさず、ここで処理を停止します（判定していないことと、ハザード区域でないことを区別するためです）。

In [ ]:
def load_hazard_geojson(file_bytes, hazard_type, label):
    """GeoJSONを読み込み、hazard_type/geometryの2列に正規化し、WGS84(EPSG:4326)へ統一する。
    Polygon/MultiPolygon以外、または空・不正なジオメトリは除外する。"""
    gdf = gpd.read_file(io.BytesIO(file_bytes))
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326)
    elif gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    before = len(gdf)
    gdf = gdf[
        gdf.geometry.notna()
        & gdf.geometry.is_valid
        & gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    ]
    dropped = before - len(gdf)
    if dropped:
        print(f"  [{label}] 空・不正、またはPolygon/MultiPolygon以外のジオメトリを{dropped}件除外しました。")

    return gpd.GeoDataFrame(
        {"hazard_type": hazard_type, "geometry": gdf.geometry.values}, crs="EPSG:4326"
    )


hazard_gdf = None

if ENABLE_HAZARD_CHECK:
    print("ハザード区域のGeoJSONファイルをアップロードしてください（複数選択可）。")
    uploaded_hazards = files.upload()

    hazard_layers = []
    for hazard_filename, hazard_bytes in uploaded_hazards.items():
        hazard_type = input(
            f"'{hazard_filename}' のハザード種別名を入力してください（例: 洪水, 土砂災害, 津波, 高潮）: "
        ).strip()
        if not hazard_type:
            hazard_type = hazard_filename
        layer = load_hazard_geojson(hazard_bytes, hazard_type, hazard_filename)
        print(f"'{hazard_filename}' を hazard_type='{hazard_type}' として読み込みました。（有効{len(layer)}件）")
        hazard_layers.append(layer)

    if hazard_layers:
        hazard_gdf = gpd.GeoDataFrame(pd.concat(hazard_layers, ignore_index=True), crs="EPSG:4326")

    if hazard_gdf is None or len(hazard_gdf) == 0:
        raise RuntimeError(
            "ENABLE_HAZARD_CHECK=True ですが、有効なハザード区域(Polygon/MultiPolygon)を1件も読み込めませんでした。"
            "GeoJSONファイルが正しくアップロードされているか、ジオメトリ形式を確認してください。"
            "ハザード判定を行わない場合は、設定セルで ENABLE_HAZARD_CHECK=False にしてください。"
        )

    print(f"ハザードデータを合計 {len(hazard_gdf)}件読み込みました。")
else:
    print("ENABLE_HAZARD_CHECK=False のため、ハザードデータの読込をスキップします。")

## 8. 有効な避難所の確認

座標が不正な避難所の行は距離計算の対象から除外します。除外件数を表示し、
有効な避難所が0件の場合はここで処理を停止します（それ以外の場合は処理を継続します）。

In [ ]:
invalid_shelter_count = int((~shelters_coord_valid).sum())
shelters_valid = shelters.loc[shelters_coord_valid].reset_index(drop=True)

print(f"避難所一覧: 全{len(shelters)}件中、座標が不正なため {invalid_shelter_count}件を除外しました。")
print(f"距離計算に使用する有効な避難所: {len(shelters_valid)}件")

if len(shelters_valid) == 0:
    raise RuntimeError(
        "有効な座標を持つ避難所が0件です。避難所一覧CSVの latitude / longitude 列を確認してください。"
    )

## 9. 避難所候補の距離計算

各要支援者について、有効な避難所すべてとの直線距離（`geopy.distance.geodesic`、メートル単位）を計算し、
近い順に `TOP_N` 件を候補として算出する関数を定義します。これは道路距離ではありません。
並び替えは丸める前の距離で行い、メートル単位への丸め（小数1桁）はCSVに出力する値を作成する際にのみ行います。
距離が同一の場合でも結果順が実行ごとにばらつかないよう、避難所名を用いて順序を安定させます。

In [ ]:
def compute_candidates(resident_lat, resident_lon, shelters_df, top_n):
    """要支援者の座標から近い順に避難所候補を [(名前, 距離m, 緯度, 経度), ...] で返す。
    座標が欠損、または緯度・経度が有効範囲外であればNoneを返す（geodesic()に不正値を渡さない）。
    距離は丸めずに返す（並び替え後、出力時にのみ丸める）。"""
    if (
        pd.isna(resident_lat)
        or pd.isna(resident_lon)
        or not (-90 <= resident_lat <= 90)
        or not (-180 <= resident_lon <= 180)
    ):
        return None

    resident_coord = (resident_lat, resident_lon)
    records = []
    for _, shelter in shelters_df.iterrows():
        shelter_coord = (shelter["latitude"], shelter["longitude"])
        distance_m = geodesic(resident_coord, shelter_coord).meters
        records.append((shelter["name"], distance_m, shelter["latitude"], shelter["longitude"]))

    records.sort(key=lambda record: (record[1], record[0]))
    return records[:top_n]

## 10. ハザード区域判定用の関数

要支援者地点・候補避難所地点がハザード区域の内部または境界上にあるか、
また要支援者と候補避難所を結ぶ直線がハザード区域と交差するかを判定する関数を定義します。

直線交差の判定は **道路上の避難経路判定ではありません**。単純に2地点を結んだ直線上にハザード区域が
存在するかどうかを確認する参考情報です。要支援者自身がハザード区域内にいる場合、直線は始点で
既に区域と交差しますが、「地点の判定」と「直線の判定」は別の列として出力するため、混同しないでください。

In [ ]:
def hazard_types_at_point(lat, lon, hazard_area):
    """座標がハザード区域の内部または境界上にあるかどうかと、該当するhazard_type（;区切り）を返す。"""
    if hazard_area is None or len(hazard_area) == 0:
        return False, ""
    if pd.isna(lat) or pd.isna(lon):
        return np.nan, np.nan

    point = Point(lon, lat)
    hit_types = sorted(hazard_area.loc[hazard_area.geometry.intersects(point), "hazard_type"].unique())
    return (len(hit_types) > 0), ";".join(hit_types)


def hazard_types_on_line(lat1, lon1, lat2, lon2, hazard_area):
    """2地点を結ぶ直線がハザード区域と交差するかどうかと、該当するhazard_typeを返す（道路経路上の判定ではない）。"""
    if hazard_area is None or len(hazard_area) == 0:
        return False, ""
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return np.nan, np.nan

    line = LineString([(lon1, lat1), (lon2, lat2)])
    hit_types = sorted(hazard_area.loc[hazard_area.geometry.intersects(line), "hazard_type"].unique())
    return (len(hit_types) > 0), ";".join(hit_types)

## 11. 候補算出とハザード判定の実行

要支援者ごとに、避難所候補・距離・（`ENABLE_HAZARD_CHECK=True` の場合のみ）ハザード判定結果を組み立てます。
距離による候補順位はハザード判定結果によって変更されません。座標がない要支援者の行も削除せず、
候補・距離を空欄のまま保持します（`match_status` 列で状態を確認できます）。

In [ ]:
result_records = []

for _, resident in residents.iterrows():
    lat, lon = resident["latitude"], resident["longitude"]
    candidates = compute_candidates(lat, lon, shelters_valid, TOP_N)

    record = {}

    if ENABLE_HAZARD_CHECK:
        in_hazard, hazard_types = hazard_types_at_point(lat, lon, hazard_gdf)
        record["resident_in_hazard"] = in_hazard
        record["resident_hazard_types"] = hazard_types

    for i in range(TOP_N):
        n = i + 1
        candidate_col = f"candidate_{n}"
        distance_col = f"distance_{n}_m"
        shelter_hazard_col = f"candidate_{n}_shelter_in_hazard"
        shelter_hazard_types_col = f"candidate_{n}_shelter_hazard_types"
        line_hazard_col = f"candidate_{n}_straight_line_intersects_hazard"
        line_hazard_types_col = f"candidate_{n}_straight_line_hazard_types"

        if candidates is not None and i < len(candidates):
            shelter_name, distance_m, shelter_lat, shelter_lon = candidates[i]
            record[candidate_col] = shelter_name
            record[distance_col] = round(distance_m, 1)

            if ENABLE_HAZARD_CHECK:
                shelter_in_hazard, shelter_hazard_types = hazard_types_at_point(
                    shelter_lat, shelter_lon, hazard_gdf
                )
                record[shelter_hazard_col] = shelter_in_hazard
                record[shelter_hazard_types_col] = shelter_hazard_types

                line_intersects, line_hazard_types = hazard_types_on_line(
                    lat, lon, shelter_lat, shelter_lon, hazard_gdf
                )
                record[line_hazard_col] = line_intersects
                record[line_hazard_types_col] = line_hazard_types
        else:
            record[candidate_col] = np.nan
            record[distance_col] = np.nan
            if ENABLE_HAZARD_CHECK:
                record[shelter_hazard_col] = np.nan
                record[shelter_hazard_types_col] = np.nan
                record[line_hazard_col] = np.nan
                record[line_hazard_types_col] = np.nan

    record["match_status"] = "ok" if candidates is not None else "no_coordinates"
    result_records.append(record)

results_df = pd.DataFrame(result_records)
final_df = pd.concat([residents.reset_index(drop=True), results_df], axis=1)

print("候補算出が完了しました。")

## 12. 結果確認

CSVを出力する前に、Notebook上で処理結果の概要と先頭数行を確認します。

In [ ]:
total_residents = len(final_df)
ok_count = int((final_df["match_status"] == "ok").sum())
no_coord_count = int((final_df["match_status"] == "no_coordinates").sum())

print(f"要支援者件数: {total_residents}件")
print(f"距離計算できた件数: {ok_count}件")
print(f"座標不足のため距離計算できなかった件数: {no_coord_count}件")
print(f"距離計算に使用した有効な避難所件数: {len(shelters_valid)}件")

if ENABLE_GEOCODING and "geocode_status" in final_df.columns:
    print("要支援者一覧のジオコーディング結果:")
    for status, count in final_df["geocode_status"].value_counts(dropna=True).items():
        print(f"  {status}: {count}件")

if ENABLE_HAZARD_CHECK:
    resident_hazard_count = int((final_df["resident_in_hazard"] == True).sum())
    print(f"ハザード区域内（境界上含む）にいる要支援者数: {resident_hazard_count}件")

    shelter_hazard_flags = [
        final_df[f"candidate_{i + 1}_shelter_in_hazard"] == True for i in range(TOP_N)
    ]
    line_hazard_flags = [
        final_df[f"candidate_{i + 1}_straight_line_intersects_hazard"] == True for i in range(TOP_N)
    ]
    shelter_hazard_count = int(pd.concat(shelter_hazard_flags, axis=1).sum().sum())
    line_hazard_count = int(pd.concat(line_hazard_flags, axis=1).sum().sum())

    print(f"ハザード区域内にある候補避難所の件数（延べ、TOP_N分の合計）: {shelter_hazard_count}件")
    print(f"候補避難所への直線がハザード区域と交差する件数（延べ、TOP_N分の合計）: {line_hazard_count}件")

display(final_df.head())

## 13. CSV出力・ダウンロード

結果をExcelで文字化けしにくい `utf-8-sig`（UTF-8 BOM付き）でCSVに出力し、ブラウザへダウンロードします。
出力CSVには個人情報が含まれ得るため、取り扱いに注意してください。

In [ ]:
final_df.to_csv(OUTPUT_FILENAME, index=False, encoding="utf-8-sig")
print(f"'{OUTPUT_FILENAME}' を出力しました。")

files.download(OUTPUT_FILENAME)